In [1]:
import joblib
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)
import sys
sys.path.append('..')

from src.predict import predict_medical_plan
from src.feature_engineering import create_features

C:\Program Files\Python312\Lib\pickle.py:1760: UserWarning: [16:36:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\gbm\../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


In [2]:
model = joblib.load(
    "../models/insurance_model.pkl"
)

FileNotFoundError: [Errno 2] No such file or directory: '../models/insurance_model.pkl'

1. Loading original cleaned dataset:

In [ ]:
df = pd.read_csv(
    "../data/processed/clean_data.csv"
)

2. Loading feature eng. function that I created:

In [ ]:
df = create_features(df)

3. Seperating target and feature variables:

In [ ]:
X = df.drop(
    columns=["medical_plan"]
)

y = df["medical_plan"]

4. Final evaluation on test set:

In [ ]:
X_test = joblib.load(
    "../models/X_test_xgb.pkl"
)

y_test = joblib.load(
    "../models/y_test_xgb.pkl"
)

In [ ]:
preds = model.predict(X_test)

5. Predictiting final score:

In [ ]:
print(
    classification_report(
        y_test,
        preds
    )
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        preds,
        average="macro"
    )
)

NameError: name 'y_test' is not defined

6. Testing single customer prediction:

In [ ]:
sample = pd.DataFrame({
    'user_id':[1001],
    'gender':['Male'],
    'age':[35],
    'state_tier':['Tier-1'],
    'occupation_class':['High-Risk'],
    'salary_bracket':['High'],
    'total_income_inr':[1200000],
    'is_smoker':[1], 
    'family_members':[4],
    'annual_expenditure_inr':[500000]
})

6.1 Applying FE:

In [ ]:
sample = create_features(sample)

sample.head()

In [ ]:
print(sample.columns)

6.2 Predict:

In [ ]:
prediction = model.predict(sample)

probabilities = model.predict_proba(sample)

prediction

6.3 Probabilities used:

In [ ]:
class_mapping = {
    0: "High",
    1: "Low",
    2: "Medium"
}

6.4 Making the mapped probabilities show actual string rather than encoded indexes:

In [ ]:
pd.DataFrame(
    probabilities,
    columns=[
        class_mapping[i]
        for i in model.classes_
    ]
)

In [ ]:
joblib.dump(
    class_mapping,
    "../backend/models/class_mapping.pkl"
)

['../backend/models/class_mapping.pkl']

7. Testing predict.py is imported correctly or not via applying on sample data:

In [ ]:
customer = {
    "user_id": 1001,
    "gender": "Male",
    "age": 32,
    "state_tier": "Tier-1",
    "occupation_class": "Professional",
    "salary_bracket": "50K-1L",
    "total_income_inr": 1200000,
    "is_smoker": 1,
    "family_members": 4,
    "annual_expenditure_inr": 500000
}

In [ ]:
predict_medical_plan(customer)

c:\Users\kadit\ML-Journey\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\ML-Journey\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


{'prediction': 'High',
 'probabilities': {'High': 0.8927, 'Low': 0.0183, 'Medium': 0.0891},
 'warnings': ['Annual expenditure is outside training range.']}

8. Production Testing On Sample Users:

8.1 Customer 1:

In [ ]:
customer_1 = {
    "user_id": 1,
    "gender": "Male",
    "age": 28,
    "state_tier": "Tier-1",
    "occupation_class": "Professional",
    "salary_bracket": "50K-1L",
    "total_income_inr": 1200000,
    "is_smoker": 1,
    "family_members": 4,
    "annual_expenditure_inr": 500000
}

In [ ]:
predict_medical_plan(customer_1)

c:\Users\kadit\ML-Journey\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\ML-Journey\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


{'prediction': 'High',
 'probabilities': {'High': 0.892, 'Low': 0.019, 'Medium': 0.089},
 'warnings': ['Annual expenditure is outside training range.']}

8.2 Customer 2:

In [ ]:
customer_2 = {
    "user_id": 2,
    "gender": "Female",
    "age": 35,
    "state_tier": "Tier-2",
    "occupation_class": "Service",
    "salary_bracket": "25K-50K",
    "total_income_inr": 600000,
    "is_smoker": 0,
    "family_members": 5,
    "annual_expenditure_inr": 300000
}

In [ ]:
predict_medical_plan(customer_2)

c:\Users\kadit\ML-Journey\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\ML-Journey\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


{'prediction': 'High',
 'probabilities': {'High': 0.8659, 'Low': 0.021, 'Medium': 0.1131},
 'warnings': []}

8.3 Customer 3:

In [ ]:
customer_3 = {
    "user_id": 3,
    "gender": "Male",
    "age": 22,
    "state_tier": "Tier-3",
    "occupation_class": "Student",
    "salary_bracket": "0-25K",
    "total_income_inr": 200000,
    "is_smoker": 0,
    "family_members": 2,
    "annual_expenditure_inr": 120000
}

8.4 Customer 4:

In [ ]:
customer_4 = {
    "user_id": 4,
    "gender": "Female",
    "age": 50,
    "state_tier": "Tier-1",
    "occupation_class": "Business",
    "salary_bracket": "1L+",
    "total_income_inr": 2500000,
    "is_smoker": 0,
    "family_members": 6,
    "annual_expenditure_inr": 900000
}

8.5 Customer 5:

In [ ]:
customer_5 = {
    "user_id": 5,
    "gender": "Male",
    "age": 65,
    "state_tier": "Tier-2",
    "occupation_class": "Retired",
    "salary_bracket": "25K-50K",
    "total_income_inr": 500000,
    "is_smoker": 1,
    "family_members": 1,
    "annual_expenditure_inr": 450000
}

In [ ]:
customers = [
    customer_1,
    customer_2,
    customer_3,
    customer_4,
    customer_5
]

for i, customer in enumerate(customers, 1):
    print(f"\nCustomer {i}")
    print(predict_medical_plan(customer))


Customer 1
{'prediction': 'High', 'probabilities': {'High': 0.892, 'Low': 0.019, 'Medium': 0.089}, 'warnings': ['Annual expenditure is outside training range.']}

Customer 2
{'prediction': 'High', 'probabilities': {'High': 0.8659, 'Low': 0.021, 'Medium': 0.1131}, 'warnings': []}

Customer 3
{'prediction': 'Low', 'probabilities': {'High': 0.0339, 'Low': 0.7283, 'Medium': 0.2378}, 'warnings': []}

Customer 4
{'prediction': 'High', 'probabilities': {'High': 0.9149, 'Low': 0.0119, 'Medium': 0.0732}, 'warnings': ['Annual expenditure is outside training range.']}

Customer 5
{'prediction': 'High', 'probabilities': {'High': 0.8474, 'Low': 0.056, 'Medium': 0.0966}, 'warnings': ['Annual expenditure is outside training range.']}


c:\Users\kadit\ML-Journey\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\ML-Journey\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\ML-Journey\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\ML-Journey\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarni

# Conclusion : Pipeline is wokring correctly!!!

In [ ]:
high_customer = (
    df[df['medical_plan']=="High"]
    .iloc[0]
    .drop('medical_plan')
    .to_dict()
)

high_customer

In [ ]:
predict_medical_plan(high_customer)

In [ ]:
print(df[['total_income_inr',
          'annual_expenditure_inr',
          'family_members',
          'age']].describe())

In [ ]:
print(df['total_income_inr'].max())
print(df['annual_expenditure_inr'].max())
print(df['family_members'].max())
print(df['age'].max())